In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# AVL Trees

This notebook implements
[AVL trees](https://en.wikipedia.org/wiki/AVL_tree).  The set $\mathcal{A}$ of *AVL trees* is defined inductively:

  - $\texttt{Nil} \in \mathcal{A}$.
  - $\texttt{Node}(k,v,l,r) \in \mathcal{A}\quad$  if 
      - $\texttt{Node}(k,v,l,r) \in \mathcal{B}$,
      - $l, r \in \mathcal{A}$, and
      - $|l.\texttt{height}() - r.\texttt{height}()| \leq 1$.

According to this definition, an AVL tree is an *ordered binary tree*
such that for every node $\texttt{Node}(k,v,l,r)$ in this tree the height of the left subtree $l$ and the right
subtree  $r$ differ at most by one. 

The class `AVLTree` represents the nodes of an AVL tree.  This class has the following member variables:

  - `mKey` is the key stored at the root of the tree,
  - `mValue` is the values associated with this key,
  - `mLeft` is the left subtree, 
  - `mRight` is the right subtree, and
  - `mHeight` is the height.

The constructor creates the empty tree. 

We also define an interface for the `AVLTree` class so TypeScript allows us to reference methods that will be implemented later.

In [ ]:
type Key = string | number;
type Value = string | number | null;

class AVLTree {
    static sNodeCount: number = 0;
    mKey: Key | null;
    mValue: Value;
    mLeft: AVLTree | null;
    mRight: AVLTree | null;
    mHeight: number;

    constructor() {
        this.mKey = null;
        this.mValue = null;
        this.mLeft = null;
        this.mRight = null;
        this.mHeight = 0;
    }
}

interface AVLTree {
    mKey: Key | null;
    mValue: Value;
    mLeft: AVLTree | null;
    mRight: AVLTree | null;
    mHeight: number;
    mID: number | null;
    isEmpty(): boolean;
    find(key: Key): Value | undefined;
    insert(key: Key, value: Value): AVLTree;
    delete(key: Key): void;
    _delMin(): [AVLTree, Key | null, Value];
    _update(t: AVLTree): void;
    _restore(): void;
    _setValues(k: Key, v: Value, l: AVLTree | null, r: AVLTree | null): void;
    _restoreHeight(): void;
    toDot(): Promise<string>;
    _assignIDs(NodeDict: Map<number, AVLTree>): void;
    maxKey(): Key | null;
}

Given an ordered binary tree $t$, the expression $t.\texttt{isEmpty}()$ checks whether $t$ is the empty tree.

In [ ]:
AVLTree.prototype.isEmpty = function(this: AVLTree): boolean {
    return this.mHeight === 0;
};

Given an ordered binary tree $t$ and a key $k$, the expression $t.\texttt{find}(k)$ returns the value stored unter the key $k$.
The method `find` is defined inductively as follows:
  - $\texttt{Nil}.\texttt{find}(k) = \Omega$,

    because the empty tree is interpreted as the empty map.
  - $\texttt{Node}(k, v, l, r).\texttt{find}(k) = v$,
      
    because the node $\texttt{Node}(k,v,l,r)$ stores the assignment $k \mapsto v$.
  - $k_1 < k_2 \rightarrow \texttt{Node}(k_2, v, l, r).\texttt{find}(k_1) = l.\texttt{find}(k_1)$,

    because if $k_1$ is less than $k_2$, then any mapping for $k_1$ has to be stored in the left subtree  $l$.
  - $k_1 > k_2 \rightarrow \texttt{Node}(k_2, v, l, r).\texttt{find}(k_1) = r.\texttt{find}(k_1)$,

    because if $k_1$ is greater than $k_2$, then any mapping for $k_1$ has to be stored in the right subtree  $r$.

In [ ]:
AVLTree.prototype.find = function(this: AVLTree, key: Key): Value | undefined {
    if (this.mKey == null) {
        return undefined;
    }     
    if (this.mKey === key) {
        return this.mValue;
    } else if (key < this.mKey) {
        if (this.mLeft == null) {
             return undefined;
        }
        return this.mLeft.find(key);
    } else {
        if (this.mRight == null) {
             return undefined;
        }
        return this.mRight.find(key);
    }
};


The method  $\texttt{insert}()$ is specified via recursive equations.
  - $\texttt{Nil}.\texttt{insert}(k,v) = \texttt{Node}(k,v, \texttt{Nil}, \texttt{Nil})$,
  - $\texttt{Node}(k, v_2, l, r).\texttt{insert}(k,v_1) = \texttt{Node}(k, v_1, l, r)$,
  - $k_1 < k_2 \rightarrow 
          \texttt{Node}(k_2, v_2, l, r).\texttt{insert}(k_1, v_1) =
          \texttt{Node}\bigl(k_2, v_2, l.\texttt{insert}(k_1,v_1), r\bigr).\texttt{restore}()$,
  - $k_1 > k_2 \rightarrow 
         \texttt{Node}(k_2, v_2, l, r).\texttt{insert}\bigl(k_1, v_1\bigr) = 
         \texttt{Node}\bigl(k_2, v_2, l, r.\texttt{insert}(k_1,v_1)\bigr).\texttt{restore}()$.

The function $\texttt{restore}$ is an auxiliary function that is defined below.  This function restores the balancing condition if it is violated after an insertion.

In [ ]:
AVLTree.prototype.insert = function(this: AVLTree, key: Key, value: Value): AVLTree {
    if (this.isEmpty()) {
        this.mKey = key;
        this.mValue = value;
        this.mLeft = new AVLTree();
        this.mRight = new AVLTree();
        this.mHeight = 1;
    } else if (this.mKey != null && this.mKey === key) {
        this.mValue = value;
    } else if (this.mKey != null && key < this.mKey) {
        if (this.mLeft) {
            this.mLeft.insert(key, value);
            this._restore();
        }
    } else {
        if (this.mRight) {
            this.mRight.insert(key, value);
            this._restore();
        }
    }
    return this;
};

The method $\texttt{self}.\texttt{delete}(k)$ removes the key $k$ from the tree $\texttt{self}$.  It is defined as follows:

  - $\texttt{Nil}.\texttt{delete}(k) = \texttt{Nil}$,
  - $\texttt{Node}(k,v,\texttt{Nil},r).\texttt{delete}(k) = r$,
  - $\texttt{Node}(k,v,l,\texttt{Nil}).\texttt{delete}(k) = l$,
  - $l \not= \texttt{Nil} \,\wedge\, r \not= \texttt{Nil} \,\wedge\, 
       \langle r',k_{min}, v_{min} \rangle := r.\texttt{delMin}()  \;\rightarrow\;
      \texttt{Node}(k,v,l,r).\texttt{delete}(k) = \texttt{Node}(k_{min},v_{min},l,r').\texttt{restore}()$
  - $k_1 < k_2 \rightarrow \texttt{Node}(k_2,v_2,l,r).\texttt{delete}(k_1) = 
    \texttt{Node}\bigl(k_2,v_2,l.\texttt{delete}(k_1),r\bigr).\texttt{restore}()$,
  - $k_1 > k_2 \rightarrow \texttt{Node}(k_2,v_2,l,r).\texttt{delete}(k_1) = 
     \texttt{Node}\bigl(k_2,v_2,l,r.\texttt{delete}(k_1)\bigr).\texttt{restore}()$.

In [ ]:
AVLTree.prototype.delete = function(this: AVLTree, key: Key): void {
    if (this.isEmpty()) {
        return;
    }
    if (this.mKey == null || this.mLeft == null || this.mRight == null) {
        return;
    }
    if (key === this.mKey) {
        if (this.mLeft.isEmpty()) {
            this._update(this.mRight);
        } else if (this.mRight.isEmpty()) {
            this._update(this.mLeft);
        } else {
            const [newRight, minKey, minValue] = this.mRight._delMin();
            this.mRight = newRight;
            this.mKey = minKey;
            this.mValue = minValue;
            this._restore();
        }
    } else if (key < this.mKey) {
        this.mLeft.delete(key);
        this._restore();
    } else {
        this.mRight.delete(key);
        this._restore();
    }
};

The method $\texttt{self}.\texttt{delMin}()$ removes the smallest key from the given tree $\texttt{self}$
and returns a triple of the form
$$ (\texttt{self}, k_m, v_m) $$
where $\texttt{self}$ is the tree that remains after removing the smallest key, while $k_m$ is the smallest key that has been found and $v_m$ is the associated value.  

The function is defined as follows:

  - $\texttt{Node}(k, v, \texttt{Nil}, r).\texttt{delMin}() = \langle r, k, v \rangle$,
  - $l\not= \texttt{Nil} \wedge \langle l',k_{min}, v_{min}\rangle := l.\texttt{delMin}() 
       \;\rightarrow\;
     \texttt{Node}(k, v, l, r).\texttt{delMin}() = 
     \langle \texttt{Node}(k, v, l', r).\texttt{restore}(), k_{min}, v_{min} \rangle
    $

In [ ]:
AVLTree.prototype._delMin = function(this: AVLTree): [AVLTree, Key | null, Value | null] {
    if (this.mLeft == null || this.mRight == null) {
        return [this, this.mKey, this.mValue];
    }
    if (this.mLeft.isEmpty()) {
        return [this.mRight, this.mKey, this.mValue];
    } else {
        const [ls, km, vm] = this.mLeft._delMin();
        this.mLeft = ls;
        this._restore();
        return [this, km, vm];
    }
};

Given two ordered binary trees $s$ and $t$, the expression $s.\texttt{update}(t)$ overwrites the attributes of $s$ with the corresponding attributes of $t$.

In [ ]:
AVLTree.prototype._update = function(this: AVLTree, t: AVLTree): void {
    this.mKey = t.mKey;
    this.mValue = t.mValue;
    this.mLeft = t.mLeft;
    this.mRight = t.mRight;
    this.mHeight = t.mHeight;
};

The function $\texttt{createNode}(k, v, l, r)$ creates an AVL-tree of that has the pair $(k, v)$ stored at its root, left subtree $l$ and right subtree $r$.

In [ ]:
function createNode(key: Key, value: Value, left: AVLTree, right: AVLTree): AVLTree {
    const node = new AVLTree();
    node.mKey = key;
    node.mValue = value;
    node.mLeft = left;
    node.mRight = right;
    node.mHeight = Math.max(left.mHeight, right.mHeight) + 1;
    return node;
}

The function $\texttt{restore}(\texttt{self})$ restores the balancing condition of the given binary tree
at the root node and recompute the variable $\texttt{mHeight}$.

The method $\texttt{restore}$ is specified via conditional equations.

  - $\texttt{Nil}.\texttt{restore}() = \texttt{Nil}$,

    because the empty tree already is an  AVL tree.
  - $|l.\texttt{height}() - r.\texttt{height}()| \leq 1 \rightarrow 
       \texttt{Node}(k,v,l,r).\texttt{restore}() = \texttt{Node}(k,v,l,r)$.

      If the balancing condition is satisfied, then nothing needs to be done. 
  - $\begin{array}[t]{cl}
              & l_1.\texttt{height}() = r_1.\texttt{height}() + 2    \\ 
       \wedge & l_1 = \texttt{Node}(k_2,v_2,l_2,r_2)                 \\
       \wedge & l_2.\texttt{height}() \geq r_2.\texttt{height}()     \\[0.2cm]
       \rightarrow & \texttt{Node}(k_1,v_1,l_1,r_1).\texttt{restore}() = 
                     \texttt{Node}\bigl(k_2,v_2,l_2,\texttt{Node}(k_1,v_1,r_2,r_1)\bigr)
       \end{array}
    $
  - $\begin{array}[t]{cl}
               & l_1.\texttt{height}() = r_1.\texttt{height}() + 2    \\ 
        \wedge & l_1 = \texttt{Node}(k_2,v_2,l_2,r_2)               \\
        \wedge & l_2.\texttt{height}() < r_2.\texttt{height}()     \\
        \wedge & r_2 = \texttt{Node}(k_3,v_3,l_3,r_3)               \\
        \rightarrow & \texttt{Node}(k_1,v_1,l_1,r_1).\texttt{restore}() = 
                      \texttt{Node}\bigl(k_3,v_3,\texttt{Node}(k_2,v_2,l_2,l_3),\texttt{Node}(k_1,v_1,r_3,r_1) \bigr)
        \end{array}
    $

  - $\begin{array}[t]{cl}
              & r_1.\texttt{height}() = l_1.\texttt{height}() + 2    \\ 
       \wedge & r_1 = \texttt{Node}(k_2,v_2,l_2,r_2)               \\
       \wedge & r_2.\texttt{height}() \geq l_2.\texttt{height}()     \\[0.2cm]
       \rightarrow & \texttt{Node}(k_1,v_1,l_1,r_1).\texttt{restore}() = 
                     \texttt{Node}\bigl(k_2,v_2,\texttt{Node}(k_1,v_1,l_1,l_2),r_2\bigr)
       \end{array}
    $
  - $\begin{array}[t]{cl}
               & r_1.\texttt{height}() = l_1.\texttt{height}() + 2    \\ 
        \wedge & r_1 = \texttt{Node}(k_2,v_2,l_2,r_2)               \\
        \wedge & r_2.\texttt{height}() < l_2.\texttt{height}()     \\
        \wedge & l_2 = \texttt{Node}(k_3,v_3,l_3,r_3)               \\
        \rightarrow & \texttt{Node}(k_1,v_1,l_1,r_1).\texttt{restore}() = 
                      \texttt{Node}\bigl(k_3,v_3,\texttt{Node}(k_1,v_1,l_1,l_3),\texttt{Node}(k_2,v_2,r_3,r_2) \bigr)
        \end{array}
    $

In [ ]:
AVLTree.prototype._restore = function(this: AVLTree): void {
    if (this.mLeft == null || this.mRight == null) return;

    if (Math.abs(this.mLeft.mHeight - this.mRight.mHeight) <= 1) {
        this._restoreHeight();
        return;
    }

    if (this.mLeft.mHeight > this.mRight.mHeight) {
        if (this.mKey == null) return; // Value darf null sein!

        const k1 = this.mKey;
        const v1 = this.mValue;
        const l1 = this.mLeft;
        const r1 = this.mRight;

        if (l1.isEmpty() || l1.mKey == null || l1.mLeft == null || l1.mRight == null) return;
        
        const k2 = l1.mKey;
        const v2 = l1.mValue;
        const l2 = l1.mLeft;
        const r2 = l1.mRight;

        if (l2.mHeight >= r2.mHeight) {
            this._setValues(k2, v2, l2, createNode(k1, v1, r2, r1));
        } else {
            if (r2.isEmpty() || r2.mKey == null || r2.mLeft == null || r2.mRight == null) return;

            const k3 = r2.mKey;
            const v3 = r2.mValue;
            const l3 = r2.mLeft;
            const r3 = r2.mRight;
            this._setValues(k3, v3, createNode(k2, v2, l2, l3), createNode(k1, v1, r3, r1));
        }
    } else {
        if (this.mKey == null) return;

        const k1 = this.mKey;
        const v1 = this.mValue;
        const l1 = this.mLeft;
        const r1 = this.mRight;

        if (r1.isEmpty() || r1.mKey == null || r1.mLeft == null || r1.mRight == null) return;

        const k2 = r1.mKey;
        const v2 = r1.mValue;
        const l2 = r1.mLeft;
        const r2 = r1.mRight;

        if (r2.mHeight >= l2.mHeight) {
            this._setValues(k2, v2, createNode(k1, v1, l1, l2), r2);
        } else {
            if (l2.isEmpty() || l2.mKey == null || l2.mLeft == null || l2.mRight == null) return;
            
            const k3 = l2.mKey;
            const v3 = l2.mValue;
            const l3 = l2.mLeft;
            const r3 = l2.mRight;
            this._setValues(k3, v3, createNode(k1, v1, l1, l3), createNode(k2, v2, r3, r2));
        }
    }
    this._restoreHeight();
};

The function $\texttt{self}.\texttt{\_setValues}(k, v, l, r)$ overwrites the member variables of the node $\texttt{self}$ with the given values.

In [ ]:
AVLTree.prototype._setValues = function(
    this: AVLTree,
    k: Key,
    v: Value,
    l: AVLTree | null,
    r: AVLTree | null
): void {
    this.mKey = k;
    this.mValue = v;
    this.mLeft = l;
    this.mRight = r;
};

In [ ]:
AVLTree.prototype._restoreHeight = function(this: AVLTree): void {
    if (this.mLeft == null || this.mRight == null) {
        return;
    }
    this.mHeight = Math.max(this.mLeft.mHeight, this.mRight.mHeight) + 1;
};

Given an ordered binary tree, this function renders the tree graphically using `Graphviz`.

In [ ]:
import { Graphviz } from "@hpcc-js/wasm";

AVLTree.prototype.toDot = async function(this: AVLTree): Promise<string> {
    AVLTree.sNodeCount = 0;
    this._assignIDs(new Map<number, AVLTree>());
    
    let dot = `digraph G {
        node [shape=record style=rounded];
    `;
    
    const nodes = new Map<number, AVLTree>();
    this._assignIDs(nodes);

    for (const [n, t] of nodes.entries()) {
        if (t.mValue != null) {
            dot += `${n} [label="{${t.mKey}|${t.mValue}}"];\n`;
        } else if (t.mKey != null) {
            dot += `${n} [label="${t.mKey}"];\n`;
        } else {
            dot += `${n} [label="" shape=point];\n`;
        }
    }

    for (const [n, t] of nodes.entries()) {
        if (t.mLeft != null) {
            dot += `${n} -> ${t.mLeft.mID};\n`;
        }
        if (t.mRight != null) {
            dot += `${n} -> ${t.mRight.mID};\n`;
        }
    }

    dot += `}`;

    return dot;
};

This method assigns a unique identifier with each node.  The dictionary `NodeDict` maps these identifiers to the nodes where they occur.

In [ ]:
AVLTree.prototype._assignIDs = function(this: AVLTree, NodeDict: Map<number, AVLTree>): void {
    AVLTree.sNodeCount += 1;
    this.mID = AVLTree.sNodeCount;
    NodeDict.set(this.mID, this);
    if (this.isEmpty()) {
        return;
    }
    if (this.mLeft != null) {
        this.mLeft._assignIDs(NodeDict);
    }
    if (this.mRight != null) {
        this.mRight._assignIDs(NodeDict);
    }
};

The function $\texttt{demo}()$ creates a small ordered binary tree.

In [ ]:
function demo(): AVLTree {
    const m = new AVLTree();
    m.insert("anton", 123);
    m.insert("hugo", 345);
    m.insert("gustav", 789);
    m.insert("jens", 234);
    m.insert("hubert", 432);
    m.insert("andre", 342);
    m.insert("philip", 342);
    m.insert("rene", 345);
    return m;
}

The function `renderTree` is used to render a AVLTree to the screen.

In [ ]:
import { display } from "tslab";

async function renderTree(t: AVLTree): Promise<void> {
    const dot = await t.toDot();
    const gv = await Graphviz.load();
    const svg = await gv.layout(dot, "svg", "dot");
    display.html(svg);
}

In [ ]:
let t = demo();
await renderTree(t);

In [ ]:
t.delete('gustav');
await renderTree(t);

In [ ]:
t.delete('hubert');
await renderTree(t);

Let's generate an ordered binary tree with random keys.

In [ ]:
let t = new AVLTree();
for (let i = 0; i < 30; i++) {
    const k = Math.floor(Math.random() * 100);
    t.insert(k, null);
}
await renderTree(t);

This tree looks more or less balanced.  Lets us try to create a tree by inserting sorted numbers because that resulted in linear complexity for ordered binary trees.

In [ ]:
let t = new AVLTree();
for (let k = 0; k < 30; k++) {
    t.insert(k, null);
}
await renderTree(t);

Next, we compute the set of prime numbers $\leq 100$.  Mathematically, this set is given as follows:
$$ \bigl\{2, \cdots, 100 \bigr\} - \bigl\{ i \cdot j \bigm| i, j \in \{2, \cdots, 100 \}\bigr\}$$

In [ ]:
const S = new AVLTree();
for (let k = 2; k <= 100; k++) {
    S.insert(k, null);
}
for (let i = 2; i <= 100; i++) {
    for (let j = 2; j <= 100; j++) {
        S.delete(i * j);
    }
}
await renderTree(S);

The function $t.\texttt{maxKey}()$ returns the biggest key in the tree $t$.  It is defined inductively:
  - $\texttt{Nil}.\texttt{maxKey}() = \Omega$,
  - $\texttt{Node}(k,v,l,\texttt{Nil}).\texttt{maxKey}() = k$,
  - $r \not= \texttt{Nil} \rightarrow \texttt{Node}(k,v,l,r).\texttt{maxKey}() = r.\texttt{maxKey}()$.

In [ ]:
AVLTree.prototype.maxKey = function(this: AVLTree): Key | null {
    if (this.isEmpty()) {
        return null;
    }
    if (this.mRight == null || this.mRight.isEmpty()) {
        return this.mKey;
    }
    return this.mRight.maxKey();
};

The function $\texttt{leanTree}(h, k)$ computes an AVL tree of height $h$ that is as lean as possible.
All key in the tree will be integers that are bigger than $k$.  The definition by induction:
  - $\texttt{leanTree}(0, k) = \texttt{Nil}$,

    because there is only one AVL tree of height $0$ and this is the tree $\texttt{Nil}$.
  - $\texttt{leanTree}(1, k) = \texttt{Node}(k+1,0,\texttt{Nil}, \texttt{Nil})$,

    since, if we abstract from the actual keys and values, there is exactly one AVL tree of height $1$.
  - $\texttt{leanTree}(h+1,k).\texttt{maxKey}() = l \rightarrow 
     \texttt{leanTree}(h+2,k) = \texttt{Node}\bigl(l+1,\,0,\,\texttt{leanTree}(h+1,k),\,\texttt{leanTree}(h, l+1)\bigr)
    $.

In [ ]:
function leanTree(h: number, k: number): AVLTree {
    if (h === 0) {
        return new AVLTree();
    }
    if (h === 1) {
        return createNode(k + 1, null, new AVLTree(), new AVLTree());
    }
    const left = leanTree(h - 1, k);
    const l = left.maxKey();
    if (typeof l !== 'number') {
        throw new Error("leanTree: maxKey returned non-number");
    }
    return createNode(
        l + 1,
        null,
        left,
        leanTree(h - 2, l + 1)
    );
}

In [ ]:
const l = leanTree(6, 0);
await renderTree(l);

In [ ]:
for (let k = 0; k < 6; k++) {
    const l = leanTree(k, 0);
    console.log(`Height ${k}:`);
    await renderTree(l);
}